# Week 1, Your First Investigation  (fully worked)

**What you'll do today.** Load a real corpus, ask it a question, and answer with a chart,
a complete investigation end to end, before any theory. The corpus is Shakespeare's 154
sonnets (1609, public domain); the second half counts real paintings the same way. Along
the way you'll set up the one routine that carries the whole course: your Drive project
folder.

The notebook has two halves. Everything through the brightness ranking runs in the
session; the sections from "Define a color" onward are the **after-class half**, about
twenty minutes, due before Week 2.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## Don't lose your work

Three rules, one minute:

1. **Save your own copy first.** Opened from GitHub, this notebook is read-only until you
   choose File → Save a copy in Drive. Do that now; edit the copy.
2. **Everything durable goes to the Drive project folder** (the cell above mounts it).
   Colab wipes its machine on idle; `/content` does not survive, your Drive does.
3. **Course updates never touch your copy.** The course notebooks improve during the
   term; to get the newest version, open it fresh from GitHub (or `git pull` if you
   cloned the repo). Your saved copy, with your work, stays yours.

## No installs today

Everything Week 1 needs ships with Colab already. (Later weeks have an install cell; today
the only setup is the Drive mount above, and, in class, putting your free Gemini key into
Colab Secrets for Week 7.)

In [ ]:
import io, os, re
import pandas as pd
import matplotlib.pyplot as plt

## Load real culture: Shakespeare's sonnets

The 154 sonnets of the 1609 Quarto, snapshotted in `data/week01/` (Project Gutenberg
#1041) so this cell needs no network. One function call and four centuries of poetry is a
list you can compute on.

In [ ]:
import re
from collections import Counter

SONNET_18 = """Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
But thy eternal summer shall not fade,
Nor shall death brag thou wander'st in his shade,
So long as men can breathe, or eyes can see,
So long lives this, and this gives life to thee."""
SONNET_130 = """My mistress' eyes are nothing like the sun;
Coral is far more red, than her lips red:
If snow be white, why then her breasts are dun;
If hairs be wires, black wires grow on her head.
I have seen roses damask'd, red and white,
But no such roses see I in her cheeks;
And in some perfumes is there more delight
Than in the breath that from my mistress reeks.
I love to hear her speak, yet well I know
That music hath a far more pleasing sound:
I grant I never saw a goddess go;
My mistress, when she walks, treads on the ground:
    And yet by heaven, I think my love as rare,
    As any she belied with false compare."""

def load_sonnets():
    raw, src = None, "built-in pair (18 and 130)"
    if os.path.exists("data/week01/shakespeare_sonnets.txt"):
        raw, src = open("data/week01/shakespeare_sonnets.txt", encoding="utf-8-sig").read(), "repo snapshot"
    else:
        try:
            import requests
            raw, src = requests.get("https://www.gutenberg.org/cache/epub/1041/pg1041.txt", timeout=30).text, "gutenberg.org"
        except Exception:
            pass
    if raw is None:
        return [SONNET_18, SONNET_130], src
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    body = re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]
    parts = re.split(r"\n\s*([IVXLC]+)\s*\n", body)
    return [parts[i+1].strip() for i in range(1, len(parts) - 1, 2)
            if 200 < len(parts[i+1].strip()) < 900], src

sonnets, src = load_sonnets()
print(len(sonnets), "sonnets loaded from the", src)
print(sonnets[17][:90], "...")

## Ask it a question, and answer with a chart

Question: **does *love* hold steady across the sequence, or does it move?** The answer is
a count per sonnet and one picture. (A rolling average smooths the line; remember that
choice; it returns in Week 2's featured fight.)

In [ ]:
per_sonnet = [len(re.findall(r"\blove\b", s.lower())) for s in sonnets]
rolling = pd.Series(per_sonnet).rolling(15, min_periods=1).mean()

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(sonnets) + 1), rolling, linewidth=2)
plt.xlabel("sonnet number"); plt.ylabel('"love" per sonnet (rolling average)')
plt.title('"love" across the sequence of 154 sonnets')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "week01_first_chart.png"), dpi=150)
plt.show()
print("chart saved to your project folder")

**Before you trust it, one question the whole course turns on:** *whose* corpus is this?
One poet, one printer's 1609 choices, one modern transcription (Project Gutenberg's), and
an editor's numbering. The chart is real; every layer between quill and CSV made choices.
Naming that isn't a gotcha. It's the skill.

## When code errors (you will need this)

The AI will sometimes hand you code that errors. The move, printed on the cheat sheet
(`kits/common-errors-cheatsheet.md`): **read the last line of the traceback, paste it back
to the AI with "this errored, fix it and tell me what went wrong," try twice, then ask a
human.** The next cell runs a bug you will actually meet and shows you its real traceback:

In [ ]:
import traceback

counts = Counter(w for s in sonnets for w in s.split())
try:
    print(counts.most_comon(5))          # <- this line looks right and is not
except Exception:
    print(traceback.format_exc())
    print("^ Read the LAST line first: it names the near-miss almost verbatim.")

In [ ]:
# The fix - method names are exact, and the traceback told us which one we meant:
print(counts.most_common(5))
print("Love:", counts["Love"], "| love:", counts["love"], " (case matters too)")

## Working with the AI assistant (guided)

In this course the AI writes most of the code; your job is to direct it and judge what
comes back. The loop, which you will run hundreds of times, has four steps:

1. **Ask precisely.** "Write a cell that counts how many of the 154 `sonnets` contain the word
   *summer*, and does the same for four more words I choose". Name the variable, name the output.
2. **Predict before you run.** One written sentence: what will this cell do? (This is the
   Week 1 check.)
3. **Run, then compare.** Did it do what you predicted? Where they differ, one of you is
   wrong, and finding out which is the learning.
4. **Interrogate.** Ask: *"explain this line by line"*, *"where would this break?"*,
   *"what is the smallest test that this did what I wanted?"* Then run that test.

Try the loop now. In Colab, open the assistant (the spark icon), give it the prompt in
step 1, and paste what it writes into the empty cell below. Write your prediction as a
comment first.

In [ ]:
# My prediction: this cell will ...
# (paste the assistant's code below, then run and compare)

# A reference version, if you want to check your result against one:
WORDS = ["summer", "winter", "day", "night", "death"]
containing = {w: sum(1 for s in sonnets if w in s.lower()) for w in WORDS}
plt.figure(figsize=(6, 3))
plt.bar(containing.keys(), containing.values())
plt.ylabel("sonnets containing the word"); plt.tight_layout(); plt.show()

**Two rules that keep the loop honest.** Never run a cell you cannot say one sentence
about, and when a cell errors, read the last line of the traceback before pasting it back
to the assistant (the section above shows the recovery routine).

## Count the whole corpus, properly

Top words next. Two lessons in one count: the stop-word list is a choice, and it has
to fit the corpus. A modern list leaves *thy* and *thou* on top.

In [ ]:
sonnet_words = [w for s in sonnets for w in re.findall(r"[a-z']+", s.lower())]
print(len(sonnet_words), "words")

STOP_MODERN = set("the of a an to is in for and on with that not as it be by so but this all no nor if or when then what which".split())
STOP_FITTED = STOP_MODERN | set("thy thou thee thine doth dost art hath shall will i my me you your his her mine their more from are was were have has can may".split())

versions = {
    "no stop list": Counter(sonnet_words),
    "modern stop list": Counter(w for w in sonnet_words if w not in STOP_MODERN and len(w) > 2),
    "stop list fit to the corpus": Counter(w for w in sonnet_words if w not in STOP_FITTED and len(w) > 2),
}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (title, c) in zip(axes, versions.items()):
    top = c.most_common(8)
    ax.barh([w for w, _ in top][::-1], [n for _, n in top][::-1], color="#7a3b2e")
    ax.set_title(title, fontsize=10)
fig.suptitle("One corpus, three answers: the stop list is a choice", y=1.04)
plt.tight_layout(); plt.show()

# Left: the/and/of, all plumbing. Middle: a modern list leaves thy and thou on top,
# because Early Modern English has its own function words. Right: with a list fit to
# the corpus, the sonnets are about what their reputation says: love by a wide margin,
# then sweet, time, beauty. The stop list is YOUR choice, renegotiated for every corpus.
STOP = STOP_FITTED
kept = [w for w in sonnet_words if w not in STOP and len(w) > 2]


## Counting pairs: n-grams

A bigram is two words in a row; a trigram, three. Counting pairs keeps a sliver of the
order the bag throws away: "my love" is a different fact than "my" and "love" counted
apart.


In [ ]:
bigrams, bigrams_kept = Counter(), Counter()
for s in sonnets:
    ws = re.findall(r"[a-z']+", s.lower())
    for a, b in zip(ws, ws[1:]):                     # each word with its neighbor
        bigrams[(a, b)] += 1
        if a not in STOP and b not in STOP and len(a) > 2 and len(b) > 2:
            bigrams_kept[(a, b)] += 1

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (title, c) in zip(axes, [("all bigrams", bigrams), ("stop words filtered", bigrams_kept)]):
    top = c.most_common(8)
    ax.barh([" ".join(p) for p, _ in top][::-1], [n for _, n in top][::-1], color="#3f6f5f")
    ax.set_title(title, fontsize=10)
fig.suptitle("Word pairs, counted", y=1.04)
plt.tight_layout(); plt.show()

tri = Counter()
for s in sonnets:
    ws = re.findall(r"[a-z']+", s.lower())
    tri.update(zip(ws, ws[1:], ws[2:]))
print("three in a row, the sparse end:")
for t, n in tri.most_common(5):
    print(f"  {n}x  {' '.join(t)}")

# Even unfiltered, the corpus's top pair is "my love". Filtered, the pairs get more
# specific: sweet self, dear love, ten times. But watch the counts shrink: the top word
# appears hundreds of times, the top pair a few dozen, the top triple six. Longer
# n-grams buy context and pay in sparsity. (She Giggles, He Gallops is a bigram count:
# "she" or "he" plus the next verb, across 2,000 screenplays.)


## The counts matrix: your first DataFrame

Put the counts in a table and you have the object every method in this course operates
on: a DataFrame. One row per sonnet, one column per word, each cell a count. Build it
for the whole vocabulary first, then cut it down: subsetting a big table is the everyday
move of data work. This table is the bag-of-words model. Tf-idf (Week 2), the classifier
(Week 3), and the embeddings (Week 5) all start from something shaped like it.


In [ ]:
# The whole table: every sonnet, every word in the vocabulary.
rows = [Counter(re.findall(r"[a-z']+", s.lower())) for s in sonnets]
dtm = pd.DataFrame(rows, index=[f"sonnet {i+1}" for i in range(len(sonnets))]).fillna(0).astype(int)
dtm = dtm[dtm.sum().sort_values(ascending=False).index]   # order columns by total count
print(dtm.shape, "(rows are sonnets, columns are every word in the corpus)")
pd.set_option("display.max_rows", 200)   # show all 154 rows; columns stay truncated
dtm


In [ ]:
# 154 rows x ~3,000 columns is the real thing, and it is mostly zeros: most words
# never appear in most sonnets. You almost never work on the whole table; you subset.

TRACK = ["love", "time", "beauty", "death", "sweet", "black"]
counts_matrix = dtm[TRACK]                 # subset columns: a list of names in brackets
print(counts_matrix.shape, "after subsetting to six words")
print(counts_matrix.head(8).to_string())

# The other three everyday subsets:
first_17 = dtm.loc["sonnet 1":"sonnet 17"]         # rows by label range (the procreation sonnets)
one_value = dtm.loc["sonnet 130", "love"]          # one row, one column: a single number
love_heavy = dtm[dtm["love"] >= 4]                 # rows by condition: sonnets with love 4+ times
print("\nsonnet 130 uses 'love'", one_value, "time;",
      len(love_heavy), "sonnets use it four or more times:", ", ".join(love_heavy.index))


In [ ]:
# Two directions to read a table.
# Down a column: one word across the whole corpus. sum() adds down columns by default.
col_sums = counts_matrix.sum()
print("column sums (each tracked word's total):")
print(col_sums.sort_values(ascending=False).to_string())

# Across a row: one sonnet's profile. axis=1 means "add across the row instead".
row_sums = counts_matrix.sum(axis=1)
print("\nrow sums, the five sonnets that use these words most:")
print(row_sums.sort_values(ascending=False).head(5).to_string())
print("\nsonnet 130's row:")
print(counts_matrix.loc["sonnet 130"].to_string())

# The DataFrame plots itself. No loops: ask the table for a chart.
col_sums.sort_values().plot(kind="barh", color="#7a3b2e", figsize=(7, 2.6),
                            title="Column sums: each tracked word's total")
plt.xlabel("count across 154 sonnets"); plt.tight_layout(); plt.show()

# The same "love" chart you built by hand earlier, in one line from the table.
counts_matrix["love"].rolling(15, min_periods=1).mean().plot(
    figsize=(8, 2.6), color="#3f6f5f", title='"love", smoothed: one column, plotted')
plt.tight_layout(); plt.show()

# Everything after this week is transformations of this table: reweight it (tf-idf),
# learn weights over its columns (classification), or replace its columns with learned
# dimensions (embeddings).


In [ ]:
# Save your tables. A CSV in your Drive project folder survives the runtime and opens
# anywhere: Excel, Sheets, next week's notebook.
dtm.to_csv(os.path.join(PROJECT_DIR, "week01_counts_full.csv"))
counts_matrix.to_csv(os.path.join(PROJECT_DIR, "week01_counts_tracked.csv"))
print("saved to", PROJECT_DIR)
print(sorted(f for f in os.listdir(PROJECT_DIR) if f.startswith("week01")))

# And the round trip: read one back, and note the index needs naming on the way in.
back = pd.read_csv(os.path.join(PROJECT_DIR, "week01_counts_tracked.csv"), index_col=0)
print("\nread back:", back.shape, "identical:", back.equals(counts_matrix))


## The bag of words

Every count so far treats a text as a bag of words: the words, with the order shaken
out. Here is the bag itself, drawn.


In [ ]:
top = Counter(kept).most_common(12)
plt.figure(figsize=(8, 3.2))
plt.bar([w for w, _ in top], [n for _, n in top], color="#7a3b2e")
plt.ylabel("count across 154 sonnets"); plt.xticks(rotation=45, ha="right")
plt.title("The bag, drawn: the corpus's most common words (stop list applied)")
plt.tight_layout(); plt.show()

## Your turn: dials to turn

Every investigation below is one edited line plus a rerun. Change the value, predict the
chart, run, compare. The same loop as with the assistant, but the question is yours.

In [ ]:
WORD_A, WORD_B = "black", "fair"    # try: time/death, day/night, sun/moon, eyes/heart

for word, style in [(WORD_A, "-"), (WORD_B, "--")]:
    per = [len(re.findall(rf"\b{word}\b", s.lower())) for s in sonnets]
    plt.plot(range(1, len(sonnets) + 1), pd.Series(per).rolling(15, min_periods=1).mean(),
             style, linewidth=2, label=word)
plt.xlabel("sonnet number"); plt.ylabel("uses per sonnet (rolling)")
plt.legend(); plt.title(f'"{WORD_A}" against "{WORD_B}" across the sequence')
plt.tight_layout(); plt.show()
# The sequence has real structure: black overtakes fair after sonnet 127, where the
# dark-lady sonnets begin. Pick a pair, predict the shape, then look.

In [ ]:
# A second dial, no new data needed: how varied is the vocabulary, sonnet by sonnet?
richness = [len(set(re.findall(r"[a-z']+", s.lower()))) / max(1, len(re.findall(r"[a-z']+", s.lower())))
            for s in sonnets]
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(sonnets) + 1), pd.Series(richness).rolling(15, min_periods=1).mean(), linewidth=2)
plt.xlabel("sonnet number"); plt.ylabel("unique words / total words (rolling)")
plt.title("Vocabulary richness across the sequence")
plt.tight_layout(); plt.show()
# Your question here: pick a claim (the late sonnets repeat themselves more; the opening
# procreation sonnets are the most formulaic), write it in one sentence, and check whether
# the chart supports it.

## Lab 2: count a real image corpus, pixel by pixel

Pictures are data the same way text is: a grid of red/green/blue numbers, so "how dark is
this painting?" is a counting question. The cell below loads real paintings from
the Met Museum's open-access collection (CC0): the course repo carries an 18-painting
snapshot in `data/week01/` (with a manifest crediting each), and the code falls back to
the live Met API, then to labeled panels, so it runs anywhere. In class the prepared
~200-thumbnail folder in your Drive plays this role; offline, colored panels stand in so the
counting code always runs.

The snapshot comes with `met_manifest.csv`, your first CSV: one row per painting, one
column per fact the Met chose to record (title, artist, date, license). What a catalog
records, and what it doesn't, is a set of choices; the session's *capta* discussion runs
on exactly this table.

In [ ]:
import io
import numpy as np
from PIL import Image

def swatch(rgb, size=96):
    arr = np.zeros((size, size, 3), dtype=np.uint8); arr[:, :] = rgb
    return Image.fromarray(arr)

def met_paintings(n=12, query="portrait", department=11):
    """Return (images, manifest): the paintings and one metadata row per painting.
    Repo snapshot first, then the live Met API; offline, colored panels stand in."""
    if os.path.exists("data/week01/met_manifest.csv"):
        man = pd.read_csv("data/week01/met_manifest.csv")
        images = {r["title"][:26]: Image.open("data/week01/" + r["file"].split("data/week01/")[-1]
                  if r["file"].startswith("data") else "data/week01/" + r["file"]).convert("RGB")
                  for _, r in man.iterrows()}
        return images, man[["objectID", "title", "artist", "date"]]
    try:
        import requests
        base = "https://collectionapi.metmuseum.org/public/collection/v1"
        ids = requests.get(f"{base}/search", timeout=20,
                           params={"hasImages": "true", "departmentId": department, "q": query}).json()["objectIDs"]
        images, rows = {}, []
        for oid in ids:
            if len(images) >= n: break
            obj = requests.get(f"{base}/objects/{oid}", timeout=20).json()
            url = obj.get("primaryImageSmall")
            if not url: continue
            try:
                img = Image.open(io.BytesIO(requests.get(url, timeout=20).content)).convert("RGB")
                img.thumbnail((140, 140))
                title = obj.get("title", "untitled")[:26]
                images[title] = img
                rows.append({"objectID": oid, "title": title,
                             "artist": obj.get("artistDisplayName", ""),
                             "date": obj.get("objectDate", "")})
            except Exception:
                continue
        if images: return images, pd.DataFrame(rows)
    except Exception as e:
        print("no network, using the stand-in panels:", e)
    fallback = {"midnight": swatch((18, 18, 42)), "storm": swatch((70, 80, 95)),
                "forest": swatch((30, 90, 50)), "terracotta": swatch((150, 80, 60)),
                "gold": swatch((205, 160, 60)), "cream": swatch((235, 225, 205))}
    man = pd.DataFrame({"objectID": 0, "title": k, "artist": "stand-in panel", "date": None}
                       for k in fallback)
    return fallback, man

corpus_images, manifest = met_paintings()
print(len(corpus_images), "images in the corpus")

# Save the metadata beside your other outputs, same habit as the word counts.
manifest.to_csv(os.path.join(PROJECT_DIR, "week01_met_manifest.csv"), index=False)
print("manifest saved to", os.path.join(PROJECT_DIR, "week01_met_manifest.csv"))


In [ ]:
def avg_brightness(img):
    a = np.asarray(img).astype(float)
    return float((0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).mean())  # perceived luminance

def avg_color(img):
    return np.asarray(img).reshape(-1, 3).mean(axis=0) / 255

ranked = sorted(corpus_images, key=lambda k: avg_brightness(corpus_images[k]))
print("darkest -> brightest:")
for k in ranked[:5]:
    print(f"  {avg_brightness(corpus_images[k]):6.1f}  {k}")

cols = 6
rows = -(-len(ranked) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(10, 2.0 * rows))
axes = np.atleast_1d(axes).ravel()
for ax in axes[len(ranked):]:
    ax.axis("off")
for ax, k in zip(axes, ranked):
    ax.imshow(corpus_images[k]); ax.axis("off")
    ax.set_title(f"{avg_brightness(corpus_images[k]):.0f}", fontsize=8)
fig.suptitle("The image corpus, ranked darkest to brightest by average luminance", fontsize=11)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 3))
plt.bar(range(len(ranked)), [avg_brightness(corpus_images[k]) for k in ranked],
        color=[avg_color(corpus_images[k]) for k in ranked])
plt.xticks(range(len(ranked)), [k[:10] for k in ranked], rotation=45, ha="right", fontsize=7)
plt.ylabel("average brightness"); plt.tight_layout(); plt.show()

## Common colors, counted

The pixel version of the word bar chart: define six named hue ranges (the boundaries are
choices; the after-class half moves them), count every pixel in every
painting, and draw the corpus's palette.

In [ ]:
import matplotlib.colors as mcolors

RANGES = {"reds": (345, 15), "oranges/browns": (15, 45), "yellows": (45, 70),
          "greens": (70, 165), "blues": (165, 255), "violets": (255, 345)}
CHART_COLORS = {"reds": "#b03a2e", "oranges/browns": "#b9722f", "yellows": "#c9a227",
                "greens": "#3f6f5f", "blues": "#2e5f8a", "violets": "#6d4b7c",
                "neutrals": "#b4b2a9"}

def color_shares(img, sat_min=0.15, val_min=0.12, val_max=0.96):
    """Share of pixels in each named hue range; unsaturated or extreme pixels are neutrals."""
    hsv = mcolors.rgb_to_hsv(np.asarray(img).astype(float) / 255)
    hue = hsv[..., 0] * 360
    colored = (hsv[..., 1] >= sat_min) & (hsv[..., 2] >= val_min) & (hsv[..., 2] <= val_max)
    shares = {}
    for name, (lo, hi) in RANGES.items():
        in_range = ((hue >= lo) & (hue < hi)) if lo < hi else ((hue >= lo) | (hue < hi))
        shares[name] = float((in_range & colored).mean())
    shares["neutrals"] = float((~colored).mean())
    return shares

palette = pd.DataFrame({name: color_shares(im) for name, im in corpus_images.items()}).T
mean_shares = palette.mean().sort_values(ascending=False)
plt.figure(figsize=(8, 3.2))
plt.bar(mean_shares.index, mean_shares.values, color=[CHART_COLORS[c] for c in mean_shares.index])
plt.ylabel("mean share of pixels"); plt.xticks(rotation=30, ha="right")
plt.title("The corpus palette: common pixel colors, counted against defined ranges")
plt.tight_layout(); plt.show()

## Missing data: the paintings' metadata

Counting pixels was the easy half. The manifest, the table describing the paintings, is
where real image corpora bite. Try a simple question: what is the average date of these
paintings?


In [ ]:
# Read the metadata back from YOUR project folder: the CSV the loader cell just saved.
manifest = pd.read_csv(os.path.join(PROJECT_DIR, "week01_met_manifest.csv"))
print(manifest[["title", "artist", "date"]].to_string(max_colwidth=38))

# The pitfall. "date" looks like a year but it is text: "ca. 1520", "1530s",
# "early 1650s". Force it to numbers and pandas quietly turns the rest into NaN.
year = pd.to_numeric(manifest["date"], errors="coerce")
print("\nmissing after naive conversion:", year.isna().sum(), "of", len(year))
print("mean year, computed on what survived:", round(year.mean(), 1))

# mean() skips NaN without telling you. That "average" used half the paintings,
# and not a random half: the circa-dated ones are mostly the oldest. The answer is
# both incomplete and biased, and nothing errored.

# The fix: pull a 4-digit year out of the text, then check what is still missing.
year_fixed = manifest["date"].astype(str).str.extract(r"(\d{4})")[0].astype(float)
print("\nmissing after extracting a year:", year_fixed.isna().sum())
print("mean year, all paintings:", round(year_fixed.mean(), 1))

# The rule: count your NaN before you average. manifest.isna().sum() is the first
# thing to run on any table you did not build yourself. Real Met records are worse
# than this snapshot: artist and date are often blank outright.


In [ ]:
YOUR_QUERY = "landscape"   # try: "flowers", "armor", "dance", "night", "saint"

my_corpus, _ = met_paintings(n=8, query=YOUR_QUERY)   # images only; the manifest half is ignored here
if os.path.exists("data/week01/met_manifest.csv"):
    print("note: the repo snapshot ignores the query; delete data/week01 or run in Colab for a live search")
my_ranked = sorted(my_corpus, key=lambda k: avg_brightness(my_corpus[k]))
fig, axes = plt.subplots(1, len(my_ranked), figsize=(1.3 * len(my_ranked), 2.4))
for ax, k in zip(np.atleast_1d(axes), my_ranked):
    ax.imshow(my_corpus[k]); ax.axis("off"); ax.set_title(f"{avg_brightness(my_corpus[k]):.0f}", fontsize=8)
fig.suptitle(f'"{YOUR_QUERY}", ranked by brightness', fontsize=10)
plt.tight_layout(); plt.show()

---

# After class: the second half (~15 minutes, before Week 2)

Two experiments that deepen what the session showed: move a color boundary and watch the
palette re-answer, then meet the largest possible bag-of-words loss. Run them slowly,
predict before running, and bring one observation to the next session.

## Move the boundary, change the answer

The color ranges above are definitions, and definitions can move. Widen "reds" by twenty
degrees and re-count: paintings change category without changing a pixel.

In [ ]:
# The boundary is the analysis. Move the oranges/reds line by twenty degrees and
# recount: paintings change color category without changing a pixel.
wide_reds = dict(RANGES); wide_reds["reds"] = (325, 35); wide_reds["oranges/browns"] = (35, 45)

def reds_share(img, ranges):
    hsv = mcolors.rgb_to_hsv(np.asarray(img).astype(float) / 255)
    hue = hsv[..., 0] * 360
    colored = (hsv[..., 1] >= 0.15) & (hsv[..., 2] >= 0.12) & (hsv[..., 2] <= 0.96)
    lo, hi = ranges["reds"]
    in_range = ((hue >= lo) & (hue < hi)) if lo < hi else ((hue >= lo) | (hue < hi))
    return float((in_range & colored).mean())

for name, im in list(corpus_images.items())[:5]:
    a, b = reds_share(im, RANGES), reds_share(im, wide_reds)
    print(f"  {name:<26} reds: {a:5.1%} -> {b:5.1%}")
# "How red is this corpus?" has no answer until you say what red is. Same lesson as the
# stop-word list, in a different medium.

In [ ]:
# The centerpiece: Sonnet 130's bag versus Sonnet 130.
s130 = next((s for s in sonnets if "nothing like the sun" in s.lower()), SONNET_130)
bag = Counter(w for w in re.findall(r"[a-z']+", s130.lower()) if w not in STOP and len(w) > 2)
print("the bag of Sonnet 130:", [w for w, _ in bag.most_common(12)])
print("\nread as a bag: a catalogue of praise (sun, coral, roses, snow, music, goddess).")
print("now read the poem:\n")
print(s130)
# Every item in that catalogue is NEGATED: nothing like the sun, no such roses, music far
# more pleasing than her voice. The bag reads a conventional love poem; Shakespeare wrote
# a satire of conventional love poems. Negation, order, and argument are exactly what the
# bag cannot hold - and this is the largest possible version of the loss.

## You did a complete investigation

Loaded four centuries of poetry, asked a question, answered it with a chart, questioned
the corpus's layers, recovered from an error methodically, and counted culture in all
three of its shapes: rows (the painting manifest), words (the sonnets), and pixels (the
paintings). The same technique each time, with a choice hiding in each.

**Sketch (homework):** browse pudding.cool and pick a favorite piece: one sentence on
what it counted, then the question from your own life it inspires. And run the
after-class half below.